# ⚽ Pipeline completo — Football Tracking → Eventos

**Cómo correrlo:** Activá GPU (Entorno de ejecución → GPU) y hacé **Ejecutar todo**.
La celda 1 instala todo y te pide **reiniciar una vez**: reiniciás y volvés a **Ejecutar todo**.
De ahí en más va solo (solo seleccionás el video cuando lo pida).

> Los *warnings* de pip sobre `pointpats/esda/spopt/...` son normales — esos paquetes no se usan.


## 0. GPU


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU — activá una en Entorno de ejecución')


## 1. Instalar (una vez) + reiniciar

Instala dependencias, actualiza ultralytics (para el modelo entrenado), deja Pillow limpio y fija
`numpy<2.1` (lo necesita el clasificador de equipos). Al terminar te pide reiniciar.


In [ ]:
import os
REPO_DIR = '/content/ncf_event_tracker'
FLAG = '/content/.setup_done'

if not os.path.exists(FLAG):
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/pipachiesa/ncf_event_tracker.git {REPO_DIR}
    !pip install -q -r {REPO_DIR}/requirements.txt
    !pip install -q filterpy scipy
    !pip install -q -U ultralytics
    !pip install -q --force-reinstall --no-cache-dir pillow
    !pip install -q 'numpy<2.1'   # para el clasificador de equipos (numba)
    open(FLAG, 'w').close()
    print('\n' + '='*66)
    print('✅ INSTALADO. Ahora: Entorno de ejecución → REINICIAR entorno,')
    print('   y volvé a Ejecutar todo (esta celda se saltea sola).')
    print('='*66)
    raise SystemExit('Reiniciá el entorno y volvé a Ejecutar todo.')

%cd {REPO_DIR}
import numpy, ultralytics
print('numpy:', numpy.__version__, '| ultralytics:', ultralytics.__version__)
try:
    import numba; from sports.common.team import TeamClassifier
    print('✅ Entorno OK — clasificador de equipos disponible.')
except Exception as e:
    print(f'⚠️  Clasificador de equipos NO disponible ({type(e).__name__}). El pipeline correrá SIN equipos (eventos con menos detalle, pero corre igual).')
assert os.path.exists('data_cleanup/main.py')


## 2. Montar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR    = '/content/drive/MyDrive/football_analytics'
TRACKING_DIR = os.path.join(DRIVE_DIR, 'tracking_output')
EVENTS_DIR   = os.path.join(DRIVE_DIR, 'event_output')
for d in (DRIVE_DIR, TRACKING_DIR, EVENTS_DIR): os.makedirs(d, exist_ok=True)
print('Resultados en:', DRIVE_DIR)


## 3. Detector

Busca tu modelo entrenado en Drive y verifica que carga. Si no hay, usa el community `football`.


In [ ]:
import glob
from ultralytics import YOLO
hits = glob.glob(os.path.join(DRIVE_DIR, 'models', '**', 'best.pt'), recursive=True)
if hits:
    DETECTOR = hits[0]
    print('Modelo entrenado:', DETECTOR)
    print('  carga OK, clases:', YOLO(DETECTOR).names)
else:
    DETECTOR = 'football'
    print('⚠️ No hay modelo entrenado en Drive — usando community \'football\'')


## 4. Subir el video

Dejá `DRIVE_VIDEO_PATH` vacío para subir desde tu compu, o poné una ruta de Drive.


In [ ]:
DRIVE_VIDEO_PATH = ''
if DRIVE_VIDEO_PATH:
    assert os.path.exists(DRIVE_VIDEO_PATH), DRIVE_VIDEO_PATH
    VIDEO_PATH = DRIVE_VIDEO_PATH
else:
    from google.colab import files
    print('Seleccioná un .mp4...')
    up = files.upload()
    VIDEO_PATH = os.path.abspath(list(up.keys())[0])
VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
print('Video:', VIDEO_PATH)


## 5. Tracking

Usa **tu modelo entrenado para jugadores** y **`football` para el balón** (cada uno donde es mejor).
Borra el CSV viejo primero, así si falla lo ves.


In [ ]:
TRACKING_CSV = os.path.join(TRACKING_DIR, VIDEO_NAME + '.csv')
if os.path.exists(TRACKING_CSV): os.remove(TRACKING_CSV)

# Cada modelo para lo suyo: el entrenado detecta mejor JUGADORES,
# pero el 'football' detecta mejor el BALÓN (78% vs 54%).
BALL_MODEL = 'football'

cmd = (
    f'python data_cleanup/main.py '
    f'--video "{VIDEO_PATH}" '
    f'--output "{TRACKING_DIR}" '
    f'--player-model "{DETECTOR}" '
    f'--ball-model "{BALL_MODEL}" '
    f'--imgsz 1280 '
    f'--pitch-imgsz 1280 '
    f'--ball-conf 0.1 '
    f'--ball-interp-gap 15 '
    f'--track-buffer 150 '
    f'--min-track-frames 12 '
    f'--pitch-model football-field '
    f'--homography-every 5'
)
print(cmd, '\n')
!{cmd}
assert os.path.exists(TRACKING_CSV), '❌ El tracking FALLÓ (no se generó el CSV). Mirá el error de arriba.'
print('✅ Tracking CSV:', TRACKING_CSV)


## 5b. ReID — fusionar IDs fragmentados

ByteTrack fragmenta a los jugadores (~187 IDs para ~25 jugadores). `data_cleanup/reid.py` fusiona
fragmentos en post-proceso usando espacio-tiempo + equipo + apariencia (histograma HSV del video),
**sin tocar el tracking**. Los eventos (celda siguiente) usan el CSV fusionado.


In [ ]:
# El ReID corre en CPU (OpenCV + numpy, cero dependencias nuevas).
RAW_TRACKING_CSV = os.path.join(TRACKING_DIR, VIDEO_NAME + '.csv')   # crudo, de la celda 5
REID_CSV = os.path.join(TRACKING_DIR, VIDEO_NAME + '_reid.csv')
if os.path.exists(REID_CSV): os.remove(REID_CSV)

# Verificación visual opcional: pinta los tracks fusionados sobre el video
# para chequear a ojo que cada color sigue al MISMO jugador entre fragmentos.
RENDER_CHECK = False

cmd = (
    f'python data_cleanup/reid.py '
    f'--tracking-csv "{RAW_TRACKING_CSV}" '
    f'--video "{VIDEO_PATH}" '
    f'--output "{REID_CSV}"'
)
if RENDER_CHECK:
    cmd += f' --render "{os.path.join(TRACKING_DIR, VIDEO_NAME + "_reid_check.mp4")}" --render-top 8'
print(cmd, '\n')
!{cmd}
assert os.path.exists(REID_CSV), '❌ El ReID FALLÓ (no se generó el CSV fusionado). Mirá el error de arriba.'
TRACKING_CSV = REID_CSV   # los eventos usan el CSV fusionado
print('✅ ReID CSV:', TRACKING_CSV)


## 7. Chequeo — ReID (IDs antes/después + 0 solapamientos)


In [ ]:
import csv

def player_ids(path):
    return {r['Object ID'] for r in csv.DictReader(open(path)) if r['Object'] == 'player'}

n_raw, n_reid = len(player_ids(RAW_TRACKING_CSV)), len(player_ids(TRACKING_CSV))
print(f'IDs de jugador: {n_raw} (crudo) -> {n_reid} (ReID)  '
      f'[{n_raw - n_reid} fusiones; baseline sin ReID era ~187]')

# 0 solapamientos: un ID fusionado nunca puede aparecer 2 veces en el mismo frame
# (eso significaría que se fusionaron dos jugadores visibles a la vez).
seen, overlaps = set(), 0
for r in csv.DictReader(open(TRACKING_CSV)):
    if r['Object'] != 'player': continue
    key = (r['Object ID'], r['Frame'])
    overlaps += key in seen
    seen.add(key)
print('Solapamientos temporales:', overlaps, '(debe ser 0)' if overlaps == 0 else '❌ HAY MERGES MALOS: bajá --tmax o subí --smin')


## 7. Chequeo — fragmentación de jugadores


In [ ]:
import csv
from collections import Counter
life = Counter()
for r in csv.DictReader(open(TRACKING_CSV)):
    if r['Object'] == 'player': life[r['Object ID']] += 1
print('IDs de jugador distintos:', len(life), ' (menos = mejor; con el modelo chico eran ~184)')


## 8. Visualización


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Circle
df = pd.read_csv(EVENTS_CSV)
print(df['Type'].value_counts())

def draw_pitch(ax):
    ax.add_patch(plt.Rectangle((0,0),1,1,fill=False,color='black',lw=2))
    ax.plot([0.5,0.5],[0,1],color='black',lw=1)
    ax.add_patch(Circle((0.5,0.5),0.083,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0.84,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
    ax.set_aspect(68.0/105.0); ax.axis('off')

pdf = df.dropna(subset=['Start X','Start Y'])
types = sorted(pdf['Type'].unique())
pal = list(plt.cm.tab10.colors)
colors = {t: pal[i % len(pal)] for i,t in enumerate(types)}
fig, ax = plt.subplots(figsize=(12,8))
ax.add_patch(plt.Rectangle((0,0),1,1,color='#3a8a3a',alpha=0.12,zorder=0)); draw_pitch(ax)
for t in types:
    s = pdf[pdf['Type']==t]
    ax.scatter(s['Start X'], s['Start Y'], s=120, color=colors[t], edgecolors='black', linewidths=0.6, label=f'{t} ({len(s)})', zorder=3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.02), ncol=4, frameon=False)
ax.set_title(f'Eventos detectados — {VIDEO_NAME}')
fig_path = os.path.join(EVENTS_DIR, VIDEO_NAME + '_event_map.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight'); plt.show()
print('Mapa:', fig_path)


## 9. Descargar los CSV


In [ ]:
from google.colab import files
files.download(TRACKING_CSV)
files.download(EVENTS_CSV)
